# Leaderboard Validation — Matrice Completa (v2)

Pipeline finale locked. Il validation seleziona; il test valuta. Il dry-run non carica modelli e non scrive artefatti scientifici.

## Leaderboard validation — matrice completa (v2)

Legge `configs/classifier_experiment_matrix.json` e le `validation_metrics.json` per job. Non apre mai un file di test. Il ranking primario e' PR-AUC; ROC-AUC e' secondario (spec 7.1).

In [ ]:
from pathlib import Path
import json, sys

def find_project_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "configs/final_generator_registry.json").is_file():
            return candidate
    raise FileNotFoundError("Project root not found")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "notebooks/utility"))
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

import classifier_metrics as cm
import finalize_validation_stage as fvs

matrix_path = PROJECT_ROOT / "configs/classifier_experiment_matrix.json"
matrix = json.loads(matrix_path.read_text()) if matrix_path.is_file() else {"jobs": []}
print(f"job totali in matrice: {len(matrix['jobs'])}")


### Ranking per architettura, per regime (controlled/full/positive-only) — spec 14.1

In [ ]:

import sys as _sys
sys.path.insert(0, str(PROJECT_ROOT / "notebooks/utility"))
from dataset_variant_registry import load_generator_registry  # noqa

dataset_registry = json.loads((PROJECT_ROOT / "configs/dataset_variant_registry.json").read_text())
variants_by_id = {v["dataset_variant_id"]: v for v in dataset_registry["variants"]}

rows = fvs.load_completed_validations(PROJECT_ROOT, stage=1)
for architecture in ("resnet50", "maxvit512", "mammofm", "raddino"):
    ranking = fvs.rank_by_generator(rows, architecture)
    print(f"\n{architecture}: {len(ranking)} generatori con almeno un seed validato")
    for entry in ranking[:5]:
        print(f"  #{entry['rank']} {entry['generator_id']}: mean_pr_auc={entry['mean_pr_auc']:.4f} (n_seeds={entry['n_seeds']})")


### Seed stability (spec 14.1)

In [ ]:

by_experiment_family = {}
for row in rows:
    key = (row["architecture"], row["dataset_variant_id"])
    by_experiment_family.setdefault(key, []).append(row["pr_auc"])
for key, values in list(by_experiment_family.items())[:10]:
    if len(values) > 1:
        print(key, cm.seed_stability(values))


### SELECTED_GENERATOR_UNION (spec 3.5 / 14.4) — validation-only, mai dal test

In [ ]:

union_payload = fvs.compute_selected_generator_union(PROJECT_ROOT, stage=1)
print(f"n_completed_jobs_considered: {union_payload['n_completed_jobs_considered']}")
print(f"SELECTED_GENERATOR_UNION: {union_payload['selected_generator_union']}")
print(f"selection_used_test_data: {union_payload['selection_used_test_data']}")
